# IIT Jodhpur — Robust Web Scraper
Scrapes: departments, academic programs, research pages, announcements,
academic regulations, newsletters, faculty profiles, course syllabi.

Features:
- SSL verification bypass + custom headers
- Automatic retries with exponential back-off
- PDF text extraction
- English-only filtering (removes Hindi/Devanagari)
- Saves cleaned corpus to `iitj_corpus.txt` and per-doc JSON

## 0 — Install dependencies

In [1]:
# Run once
!pip install requests beautifulsoup4 lxml pdfplumber tqdm langdetect urllib3 --quiet

## 1 — Imports & Session Setup

In [2]:
import os, re, json, time, random, hashlib, io, warnings
from pathlib import Path
from urllib.parse import urljoin, urlparse
from collections import defaultdict

import requests
import urllib3
from bs4 import BeautifulSoup
import pdfplumber
from tqdm.notebook import tqdm

# ── Silence SSL warnings (we bypass verification intentionally) ───────────────
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
warnings.filterwarnings('ignore')

# ── Output directory ──────────────────────────────────────────────────────────
OUT_DIR = Path("iitj_corpus")
OUT_DIR.mkdir(exist_ok=True)
(OUT_DIR / "pdfs").mkdir(exist_ok=True)

print("Setup complete. Output directory:", OUT_DIR.resolve())

Setup complete. Output directory: D:\nlu\iitj_corpus


## 2 — HTTP Session with Retries & SSL Bypass

In [3]:
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# ── Retry strategy ────────────────────────────────────────────────────────────
RETRY_STRATEGY = Retry(
    total             = 4,
    backoff_factor    = 1.5,          # waits 1.5, 3, 6, 12 seconds
    status_forcelist  = [429, 500, 502, 503, 504],
    allowed_methods   = ["HEAD", "GET", "OPTIONS"],
    raise_on_status   = False,
)

def make_session() -> requests.Session:
    """Return a session with retries, SSL bypass, and realistic headers."""
    session = requests.Session()
    adapter = HTTPAdapter(max_retries=RETRY_STRATEGY)
    session.mount("https://", adapter)
    session.mount("http://",  adapter)
    session.verify = False          # bypass broken IIT Jodhpur SSL certs
    session.headers.update({
        "User-Agent"      : ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                             "AppleWebKit/537.36 (KHTML, like Gecko) "
                             "Chrome/124.0.0.0 Safari/537.36"),
        "Accept"          : "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Accept-Language" : "en-US,en;q=0.9",
        "Accept-Encoding" : "gzip, deflate, br",
        "Connection"      : "keep-alive",
        "DNT"             : "1",
    })
    return session

SESSION = make_session()
print("HTTP session ready (SSL verify=False, 4 retries with back-off).")

HTTP session ready (SSL verify=False, 4 retries with back-off).


## 3 — URL Catalogue

In [4]:
URL_CATALOGUE = [
    # ── MANDATORY DOCUMENTS (High-priority for Task-1) ──────────────────────
    ("pdf_regulation", "UG_PG_Ordinance", "https://iitj.ac.in/PageImages/Gallery/02-2025/ACADEMIC-PROGRAMMES-638755723291553278.pdf"),
    ("academics", "academic_home", "https://iitj.ac.in/office-of-academics/en/academics"),
    ("news", "announcements", "https://www.iitj.ac.in/main/en/all-announcement"),
    ("BTech", "Bioscience_Bioengineering", "https://www.iitj.ac.in/bioscience-bioengineering/en/undergraduate-program"),
    ("BTech", "Computer_Science_Engineering", "https://www.iitj.ac.in/computer-science-engineering/en/undergraduate-programs"),
    ("BTech", "Electrical_Engineering", "https://www.iitj.ac.in/electrical-engineering/en/undergraduate-program"),
    ("BTech", "Mechanical_Engineering", "https://www.iitj.ac.in/mechanical-engineering/en/undergraduate-program"),
    ("BTech", "Chemical_Engineering", "https://www.iitj.ac.in/chemical-engineering/en/undergraduate-program"),
    ("BTech", "Civil_and_Infrastructure_Engineering", "https://www.iitj.ac.in/civil-and-infrastructure-engineering/en/btech"),
    
    


    # ── DEPARTMENTS (10) ──────────────────────────────────────────────────────
    ("dept", "BB", "https://www.iitj.ac.in/bioscience-bioengineering"),
    ("dept", "CH", "https://www.iitj.ac.in/chemistry"),
    ("dept", "CHE", "https://www.iitj.ac.in/chemical-engineering"),
    ("dept", "CE", "https://www.iitj.ac.in/civil-and-infrastructure-engineering"),
    ("dept", "CSE", "https://www.iitj.ac.in/computer-science-engineering"),
    ("dept", "EE", "https://www.iitj.ac.in/electrical-engineering"),
    ("dept", "MA", "https://www.iitj.ac.in/mathematics"),
    ("dept", "ME", "https://www.iitj.ac.in/mechanical-engineering"),
    ("dept", "MME", "https://www.iitj.ac.in/materials-engineering"),
    ("dept", "PH", "https://www.iitj.ac.in/physics"),

    # ── SCHOOLS (4) ──────────────────────────────────────────────────────────
    ("school", "SAIDE", "https://www.iitj.ac.in/school-of-artificial-intelligence-data-science"),
    ("school", "SOD", "https://www.iitj.ac.in/school-of-design"),
    ("school", "SOLA", "https://www.iitj.ac.in/school-of-liberal-arts"),
    ("school", "SME", "https://www.iitj.ac.in/schools/en/School-of-Management-&-Entrepreneurship"),

    # ── CENTRES (7) ───────────────────────────────────────────────────────────
    ("centre", "CET", "https://www.iitj.ac.in/cete"),
    ("centre", "CETSD", "https://www.iitj.ac.in/cetsd"),
    ("centre", "CRF", "https://www.iitj.ac.in/crf"),
    ("centre", "CTFP", "https://iitj.ac.in/center-for-technology-foresight-and-policy/"),
    ("centre", "MCOENSSR", "https://iitj.ac.in/manekshaw-centre/en/Manekshaw-Centre"),
    ("centre", "MedicalTech", "https://iitj.ac.in/medical-technologies/en/medical-technologies"),
    ("centre", "RCRICE", "https://www.iitj.ac.in/rcrice"),

    # ── IDRPs / IDRCs (6) ────────────────────────────────────────────────────
    ("idrp", "AIoT", "https://iitj.ac.in/iot"),
    ("idrp", "DigitalHealth", "https://www.iitj.ac.in/cdh"),
    ("idrp", "DigitalHumanities", "https://www.iitj.ac.in/dh"),
    ("idrp", "Quantum", "https://www.iitj.ac.in/qic"),
    ("idrp", "Robotics", "https://www.iitj.ac.in/rm"),
    ("idrp", "SpaceScience", "https://www.iitj.ac.in/sst"),
]

print(f"Total URLs in catalogue: {len(URL_CATALOGUE)}")
cat_counts = defaultdict(int)
for cat, *_ in URL_CATALOGUE:
    cat_counts[cat] += 1
for cat, cnt in sorted(cat_counts.items()):
    print(f"  {cat:<20s} {cnt} URLs")

Total URLs in catalogue: 36
  BTech                6 URLs
  academics            1 URLs
  centre               7 URLs
  dept                 10 URLs
  idrp                 6 URLs
  news                 1 URLs
  pdf_regulation       1 URLs
  school               4 URLs


## 4 — Core Fetching & Parsing Utilities

In [5]:
# ── English detection (reject Devanagari / other scripts) ─────────────────────
DEVANAGARI_RE = re.compile(r'[\u0900-\u097F]+')

def remove_non_english(text: str) -> str:
    """
    Remove lines/sentences that are predominantly non-English.
    Strategy: drop any line containing Devanagari characters,
    then strip residual non-ASCII noise.
    """
    lines = text.splitlines()
    clean = []
    for line in lines:
        if DEVANAGARI_RE.search(line):      # skip Hindi / Sanskrit lines
            continue
        # Remove isolated non-ASCII characters (Chinese, Arabic, etc.)
        line = re.sub(r'[^\x00-\x7F]+', ' ', line)
        line = line.strip()
        if line:
            clean.append(line)
    return '\n'.join(clean)


# ── Generic HTML text extractor ───────────────────────────────────────────────
BOILERPLATE_TAGS = [
    'script', 'style', 'nav', 'footer', 'header',
    'aside', 'noscript', 'iframe', 'form',
]

def extract_html_text(html: str, url: str = '') -> str:
    """
    Parse HTML, strip boilerplate tags, return clean English text.
    """
    soup = BeautifulSoup(html, 'lxml')

    # Drop boilerplate
    for tag in soup(BOILERPLATE_TAGS):
        tag.decompose()

    # Extract visible text
    text = soup.get_text(separator=' ', strip=True)

    # Collapse whitespace
    text = re.sub(r'[ \t]{2,}', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)

    # Remove URLs, emails
    text = re.sub(r'https?://\S+', '', text)
    text = re.sub(r'\S+@\S+\.\S+', '', text)

    # English-only filter
    text = remove_non_english(text)

    return text.strip()


# ── PDF text extractor ────────────────────────────────────────────────────────
def extract_pdf_text(pdf_bytes: bytes) -> str:
    """
    Extract text from PDF bytes using pdfplumber.
    Falls back to empty string on failure.
    """
    try:
        text_parts = []
        with pdfplumber.open(io.BytesIO(pdf_bytes)) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    text_parts.append(page_text)
        text = '\n'.join(text_parts)
        return remove_non_english(text)
    except Exception as e:
        print(f"    [PDF parse error] {e}")
        return ''


# ── Unified fetch function ─────────────────────────────────────────────────────
def fetch_url(
    url      : str,
    category : str,
    label    : str,
    timeout  : int  = 20,
    delay    : float = 0.8,
) -> dict:
    """
    Fetch one URL (HTML or PDF) and return a result dict.
    Sleeps `delay` seconds after the request (polite crawling).
    """
    result = {
        'url'     : url,
        'category': category,
        'label'   : label,
        'status'  : 'pending',
        'text'    : '',
        'chars'   : 0,
    }
    try:
        resp = SESSION.get(url, timeout=timeout)
        resp.raise_for_status()

        content_type = resp.headers.get('Content-Type', '').lower()

        if 'pdf' in content_type or url.lower().endswith('.pdf'):
            text = extract_pdf_text(resp.content)
        else:
            text = extract_html_text(resp.text, url)

        result['text']   = text
        result['chars']  = len(text)
        result['status'] = 'ok' if len(text) > 100 else 'empty'

    except requests.exceptions.SSLError as e:
        result['status'] = f'ssl_error: {str(e)[:80]}'
    except requests.exceptions.ConnectionError as e:
        result['status'] = f'conn_error: {str(e)[:80]}'
    except requests.exceptions.Timeout:
        result['status'] = 'timeout'
    except requests.exceptions.HTTPError as e:
        result['status'] = f'http_{resp.status_code}'
    except Exception as e:
        result['status'] = f'error: {str(e)[:80]}'

    time.sleep(delay + random.uniform(0, 0.5))  # polite delay
    return result


print("Utilities defined.")

Utilities defined.


## 5 — Faculty Profile Deep-Scraper

In [6]:
# ── Verified 2026 Faculty Listing URLs ────────────────────────────────────────
FACULTY_LISTING_URLS = [
    # DEPARTMENTS (10)
    ("faculty", "BB",   "https://www.iitj.ac.in/People/List?dept=bioscience-bioengineering&c=1c8346b0-07a4-401e-bce0-7f571bb4fabd"),
    ("faculty", "CHE",  "https://www.iitj.ac.in/People/List?dept=chemical-engineering&c=1c8346b0-07a4-401e-bce0-7f571bb4fabd"),
    ("faculty", "CH",   "https://www.iitj.ac.in/People/List?dept=chemistry&c=1c8346b0-07a4-401e-bce0-7f571bb4fabd"),
    ("faculty", "CE",   "https://www.iitj.ac.in/People/List?dept=civil-and-infrastructure-engineering&c=1c8346b0-07a4-401e-bce0-7f571bb4fabd"),
    ("faculty", "CSE",  "https://www.iitj.ac.in/People/List?dept=computer-science-engineering&c=1c8346b0-07a4-401e-bce0-7f571bb4fabd"),
    ("faculty", "EE",   "https://www.iitj.ac.in/People/List?dept=electrical-engineering&c=1c8346b0-07a4-401e-bce0-7f571bb4fabd"),
    ("faculty", "HSS",  "https://www.iitj.ac.in/People/List?dept=humanities-and-social-sciences&c=1c8346b0-07a4-401e-bce0-7f571bb4fabd"),
    ("faculty", "MME",  "https://www.iitj.ac.in/People/List?dept=materials-engineering&c=1c8346b0-07a4-401e-bce0-7f571bb4fabd"),
    ("faculty", "MA",   "https://www.iitj.ac.in/People/List?dept=mathematics&c=1c8346b0-07a4-401e-bce0-7f571bb4fabd"),
    ("faculty", "ME",   "https://www.iitj.ac.in/People/List?dept=mechanical-engineering&c=1c8346b0-07a4-401e-bce0-7f571bb4fabd"),
    ("faculty", "PH",   "https://www.iitj.ac.in/People/List?dept=physics&c=1c8346b0-07a4-401e-bce0-7f571bb4fabd"),

    # SCHOOLS (4)
    ("faculty", "SAIDE", "https://www.iitj.ac.in/People/List?dept=school-of-artificial-intelligence-data-science&c=1c8346b0-07a4-401e-bce0-7f571bb4fabd"),
    ("faculty", "SME",   "https://www.iitj.ac.in/People/List?dept=schools&c=1c8346b0-07a4-401e-bce0-7f571bb4fabd"),
    ("faculty", "SOD",   "https://www.iitj.ac.in/People/List?dept=school-of-design&c=1c8346b0-07a4-401e-bce0-7f571bb4fabd"),
    ("faculty", "SOLA",  "https://www.iitj.ac.in/People/List?dept=school-of-liberal-arts&c=1c8346b0-07a4-401e-bce0-7f571bb4fabd"),
]

def extract_faculty_profile_links(html: str, base_url: str) -> list:
    """Find individual faculty profile links. Modified to be more aggressive."""
    soup = BeautifulSoup(html, 'lxml')
    links = set()
    for a in soup.find_all('a', href=True):
        href = a['href']
        # The new website uses '/People/Profile?id=...' or '/People/List?dept=...'
        # We target links that look like individual profiles.
        if 'Profile?' in href or '/People/' in href:
            full = urljoin(base_url, href)
            # Filter to only keep links on the iitj.ac.in domain
            if 'iitj.ac.in' in urlparse(full).netloc:
                links.add(full)
    return list(links)

def scrape_faculty_profiles(listing_urls: list, max_profiles_per_dept: int = 50) -> list:
    all_results = []
    for (cat, label, url) in listing_urls:
        print(f"\n  Checking Department: {label}")
        try:
            # 1. Fetch the directory page
            resp = SESSION.get(url, timeout=20)
            resp.raise_for_status()
            profile_links = extract_faculty_profile_links(resp.text, url)
            
            if profile_links:
                print(f"    [FOUND SAMPLE]: {profile_links[0]}")
            
            # 2. Scrape individual profiles
            profile_links = profile_links[:max_profiles_per_dept]
            for plink in profile_links:
                # IMPORTANT: Set 'Referer' to the department listing URL
                # This tricks the server into thinking we are a real user clicking a link
                custom_headers = {
                    "Referer": url,
                    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0"
                }
                
                # Use SESSION.get directly here to ensure headers are applied
                p_resp = SESSION.get(plink, headers=custom_headers, timeout=15)
                
                if p_resp.status_code == 200 and len(p_resp.text) > 200:
                    # Process the successful text
                    from bs4 import BeautifulSoup
                    soup = BeautifulSoup(p_resp.text, 'lxml')
                    clean_text = soup.get_text(separator=' ')
                    
                    all_results.append({
    'url': plink,
    'category': 'faculty',   # ✅ ADD THIS LINE
    'label': f"faculty_{label}",
    'text': clean_text,
    'chars': len(clean_text),
    'status': 'ok'
})
                    print(f"      ✓ Scraped: {plink[-15:]}... ({len(clean_text)} chars)")
                else:
                    print(f"      ✗ Failed: {plink[-15:]}... [Status: {p_resp.status_code}]")
                    
        except Exception as e:
            print(f"    [ERROR] {label}: {e}")
    return all_results

print("Updated faculty scraper defined. Run the next cell to start scraping.")

Updated faculty scraper defined. Run the next cell to start scraping.


## 6 — Announcement / Notice Link Expander

In [7]:
NOTICE_INDEX_URLS = [
    "https://www.iitj.ac.in/main/en/all-announcement",
    "https://www.iitj.ac.in/electrical-engineering/en/News-Newsletter?dsl=2&dslID=fa6744ca-a41f-470e-88ba-8249b8c6c978",
    "https://iitj.ac.in/mathematics/en/Announcement-Page?dsl=2&dslID=d3e97f68-578c-4aef-8a00-3f97feecafa9",
    "https://www.iitj.ac.in/office-of-academics/en/academics" # Contains syllabus/regulation circulars
]

def expand_notice_links(
    index_urls : list,
    base       : str = "https://iitj.ac.in",
    max_links  : int = 300,
) -> list:
    """
    Crawl notice index pages and return up to `max_links` individual
    notice detail URLs (HTML) or PDF links.
    """
    found = []
    for url in index_urls:
        try:
            resp = SESSION.get(url, timeout=15)
            resp.raise_for_status()
            soup = BeautifulSoup(resp.text, 'lxml')
            for a in soup.find_all('a', href=True):
                href = a['href']
                full = urljoin(base, href)
                # Accept detail pages or PDFs
                if (
                    full.startswith(base) and
                    full not in [u for _, _, u in URL_CATALOGUE] and
                    full not in found and
                    len(full) > len(base) + 5
                ):
                    found.append(full)
            if len(found) >= max_links:
                break
        except Exception as e:
            print(f"  [WARN] {url}: {e}")
    return found[:max_links]

print("Notice expander defined.")

Notice expander defined.


## 7 — Main Crawl

In [8]:
all_docs = []   # list of result dicts

# ── 7-A  Crawl primary URL catalogue ─────────────────────────────────────────
print("=" * 60)
print(" PHASE 1: Primary URL Catalogue")
print("=" * 60)

for (cat, label, url) in tqdm(URL_CATALOGUE, desc="Primary URLs"):
    r = fetch_url(url, cat, label)
    status_sym = "✓" if r['status'] == 'ok' else "✗"
    print(f"  {status_sym} [{cat:<18s}] {label:<30s} {r['chars']:>7,} chars  [{r['status']}]")
    all_docs.append(r)

ok_count = sum(1 for d in all_docs if d['status'] == 'ok')
print(f"\nPhase 1 done: {ok_count}/{len(URL_CATALOGUE)} URLs returned text.")

 PHASE 1: Primary URL Catalogue


Primary URLs:   0%|          | 0/36 [00:00<?, ?it/s]

  ✓ [pdf_regulation    ] UG_PG_Ordinance                 97,342 chars  [ok]
  ✓ [academics         ] academic_home                    2,110 chars  [ok]
  ✓ [news              ] announcements                    1,488 chars  [ok]
  ✓ [BTech             ] Bioscience_Bioengineering        8,726 chars  [ok]
  ✓ [BTech             ] Computer_Science_Engineering     4,483 chars  [ok]
  ✓ [BTech             ] Electrical_Engineering           9,776 chars  [ok]
  ✓ [BTech             ] Mechanical_Engineering           7,923 chars  [ok]
  ✓ [BTech             ] Chemical_Engineering             5,111 chars  [ok]
  ✓ [BTech             ] Civil_and_Infrastructure_Engineering   1,747 chars  [ok]
  ✓ [dept              ] BB                               5,105 chars  [ok]
  ✓ [dept              ] CH                               4,828 chars  [ok]
  ✓ [dept              ] CHE                              4,372 chars  [ok]
  ✓ [dept              ] CE                               5,784 chars  [ok]
  ✓ [d

In [9]:
# ── 7-B  Expand & crawl notice / announcement links ───────────────────────────
print("\n" + "=" * 60)
print(" PHASE 2: Notice / Announcement Expansion")
print("=" * 60)

notice_links = expand_notice_links(NOTICE_INDEX_URLS, max_links=30)
print(f"Found {len(notice_links)} notice links to crawl.")

for i, url in enumerate(tqdm(notice_links, desc="Notice pages"), 1):
    r = fetch_url(url, 'notice', f'notice_{i:03d}')
    if r['status'] == 'ok':
        print(f"  ✓ notice_{i:03d}  {url[:70]}  ({r['chars']:,} chars)")
        all_docs.append(r)


 PHASE 2: Notice / Announcement Expansion
Found 30 notice links to crawl.


Notice pages:   0%|          | 0/30 [00:00<?, ?it/s]

  ✓ notice_001  https://iitj.ac.in/Main/en/Help  (1,835 chars)
  ✓ notice_002  https://iitj.ac.in/Search?lg=en  (659 chars)
  ✓ notice_004  https://iitj.ac.in/AtoZ?lg=en  (76,226 chars)
  ✓ notice_005  https://iitj.ac.in/main/en/iitj  (3,667 chars)
  ✓ notice_006  https://iitj.ac.in/m/Index/main-institute?lg=en  (754 chars)
  ✓ notice_007  https://iitj.ac.in/m/Index/main-statutory bodies?lg=en  (773 chars)
  ✓ notice_008  https://iitj.ac.in/m/Index/main-key functionaries?lg=en  (817 chars)
  ✓ notice_009  https://iitj.ac.in/m/Index/main-infrastructure?lg=en  (734 chars)
  ✓ notice_010  https://iitj.ac.in/m/Index/main-outreach?lg=en  (855 chars)
  ✓ notice_011  https://iitj.ac.in/office-of-director/en/office-of-director  (3,578 chars)
  ✓ notice_012  https://iitj.ac.in/office-of-deputy-director/en/office-of-deputy-direc  (1,148 chars)
  ✓ notice_013  https://iitj.ac.in/office-of-registrar/en/office-of-registrar  (1,735 chars)
  ✓ notice_014  https://iitj.ac.in/office-of-administration/e

In [10]:
# ── 7-C  Faculty profiles ─────────────────────────────────────────────────────
print("\n" + "=" * 60)
print(" PHASE 3: Faculty Profile Pages")
print("=" * 60)

faculty_docs = scrape_faculty_profiles(
    FACULTY_LISTING_URLS,
    max_profiles_per_dept=58,   # adjust as needed
)
all_docs.extend(faculty_docs)
print(f"\nFaculty profiles collected: {len(faculty_docs)}")


 PHASE 3: Faculty Profile Pages

  Checking Department: BB
    [FOUND SAMPLE]: https://www.iitj.ac.in/People/List?dept=bioscience-bioengineering&c=1c8346b0-07a4-401e-bce0-7f571bb4fabd&ln=hi
      ✓ Scraped: 71bb4fabd&ln=hi... (15684 chars)
      ✓ Scraped: 89-1333f53ab502... (5572 chars)
      ✓ Scraped: 4b-645b450b32b0... (5030 chars)
      ✓ Scraped: c3-e0ccbda9eff6... (6614 chars)
      ✓ Scraped: 2a-35d7180f912b... (5399 chars)
      ✓ Scraped: 7a-9504b80c2dd7... (6008 chars)
      ✓ Scraped: 47-8488def6c02a... (6246 chars)
      ✓ Scraped: 72-176a5b7ebbca... (5803 chars)
      ✓ Scraped: 4b-56d9cb7d7c13... (6234 chars)
      ✓ Scraped: ef-1ef9a327fbe2... (6280 chars)
      ✓ Scraped: c8-58dafdc29fbb... (5590 chars)
      ✓ Scraped: 55-2ec5ff1cad0e... (8485 chars)
      ✓ Scraped: cc-03c8be076400... (6989 chars)
      ✓ Scraped: d1-224d0199e231... (5706 chars)
      ✓ Scraped: 27-90e07b057c4d... (5553 chars)
      ✓ Scraped: b4-d4b93907a615... (5419 chars)
      ✓ Scraped: 92-2925

## 8 — Corpus Statistics

In [11]:
ok_docs   = [d for d in all_docs if d['status'] == 'ok']
fail_docs = [d for d in all_docs if d['status'] != 'ok']

total_chars = sum(d['chars'] for d in ok_docs)

print("=" * 55)
print(f"  Total URLs attempted   : {len(all_docs)}")
print(f"  Successfully scraped   : {len(ok_docs)}")
print(f"  Failed / empty         : {len(fail_docs)}")
print(f"  Total characters       : {total_chars:,}")
print(f"  Avg chars per doc      : {total_chars // max(len(ok_docs),1):,}")
print("=" * 55)

print("\n--- Per-category summary ---")
cat_stats = defaultdict(lambda: {'ok': 0, 'fail': 0, 'chars': 0})
for d in all_docs:
    cat = d.get('category', 'uncategorized')
    if d['status'] == 'ok':
        cat_stats[cat]['ok']    += 1
        cat_stats[cat]['chars'] += d['chars']
    else:
        cat_stats[cat]['fail']  += 1

print(f"{'Category':<22} {'OK':>4} {'Fail':>5} {'Chars':>12}")
print("-" * 47)
for cat, s in sorted(cat_stats.items()):
    print(f"{cat:<22} {s['ok']:>4} {s['fail']:>5} {s['chars']:>12,}")

print("\n--- Failed URLs ---")
for d in fail_docs:
    print(f"  [{d['status'][:40]:<40}] {d['url'][:70]}")

  Total URLs attempted   : 351
  Successfully scraped   : 351
  Failed / empty         : 0
  Total characters       : 2,143,643
  Avg chars per doc      : 6,107

--- Per-category summary ---
Category                 OK  Fail        Chars
-----------------------------------------------
BTech                     6     0       37,766
academics                 1     0        2,110
centre                    7     0       16,940
dept                     10     0       50,398
faculty                 286     0    1,769,516
idrp                      6     0       17,450
news                      1     0        1,488
notice                   29     0      134,519
pdf_regulation            1     0       97,342
school                    4     0       16,114

--- Failed URLs ---


## 9 — Save Corpus

In [ ]:
# ── 9-A  Save all docs as JSON (metadata + text) ──────────────────────────────
json_path = OUT_DIR / 'all_docs.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(ok_docs, f, ensure_ascii=False, indent=2)
print(f"Saved {len(ok_docs)} docs → {json_path}")

# ── 9-B  Save flat text corpus ────────────────────────────────────────────────
corpus_path = OUT_DIR / 'corpus.txt'
with open(corpus_path, 'w', encoding='utf-8') as f:
    for doc in ok_docs:
        f.write(f"\n\n### SOURCE: {doc.get('category','uncategorized')} | {doc.get('label','unknown')} ###\n")
        f.write(doc['text'])
print(f"Saved flat corpus      → {corpus_path}  ({corpus_path.stat().st_size/1024:.1f} KB)")

# ── 9-C  Per-category text files ─────────────────────────────────────────────
for cat in cat_stats:
    cat_docs = [d for d in ok_docs if d.get('category') == cat]
    if not cat_docs:
        continue
    cat_path = OUT_DIR / f"{cat}.txt"
    with open(cat_path, 'w', encoding='utf-8') as f:
        for doc in cat_docs:
            f.write(f"\n\n### {doc['label']} | {doc['url']} ###\n")
            f.write(doc['text'])
    print(f"  {cat:<22} → {cat_path.name}  ({cat_path.stat().st_size/1024:.1f} KB)")

print("\nAll files saved in:", OUT_DIR.resolve())

Saved 351 docs → iitj_corpus\all_docs.json
Saved flat corpus      → iitj_corpus\iitj_corpus.txt  (2364.6 KB)
  pdf_regulation         → pdf_regulation.txt  (96.8 KB)
  academics              → academics.txt  (2.1 KB)
  news                   → news.txt  (1.5 KB)
  BTech                  → BTech.txt  (37.6 KB)
  dept                   → dept.txt  (49.8 KB)
  school                 → school.txt  (16.0 KB)
  centre                 → centre.txt  (17.0 KB)
  idrp                   → idrp.txt  (17.3 KB)
  notice                 → notice.txt  (134.7 KB)
  faculty                → faculty.txt  (2010.8 KB)

All files saved in: D:\nlu\iitj_corpus


## 10 — Quick Sanity Check

In [13]:
from collections import Counter
import re

# Concatenate all text
full_text = '\n'.join(d['text'] for d in ok_docs)

# Simple token count (no NLTK needed)
raw_tokens = re.findall(r'[a-zA-Z]{2,}', full_text.lower())

print(f"Total raw tokens  : {len(raw_tokens):,}")
print(f"Unique words       : {len(set(raw_tokens)):,}")
print("\nTop-30 tokens:")
for w, c in Counter(raw_tokens).most_common(30):
    print(f"  {w:<20s} {c}")

Total raw tokens  : 142,682
Unique words       : 8,647

Top-30 tokens:
  the                  4236
  of                   4222
  and                  4092
  in                   3261
  for                  1801
  to                   1721
  jodhpur              1679
  department           1618
  research             1548
  at                   1440
  iitj                 1380
  engineering          1368
  iit                  1315
  dot                  1138
  institute            1090
  program              1019
  is                   966
  school               938
  email                906
  web                  862
  technology           790
  contact              738
  by                   733
  he                   707
  links                676
  portal               675
  feedback             672
  on                   656
  as                   648
  members              647


In [14]:
# Preview first 500 chars of each category
for cat in sorted(cat_stats):
    docs = [d for d in ok_docs if d['category'] == cat]
    if docs:
        snippet = docs[0]['text'][:300].replace('\n', ' ')
        print(f"\n[{cat}] — {docs[0]['label']}")
        print(f"  {snippet}…")


[BTech] — Bioscience_Bioengineering
  Undergraduate Program | Bioscience & Bioengineering | IIT Jodhpur ###147852369$$$_RedirectToLoginPage_%%%963258741!!! Undergraduate Program B. Tech in Bioengineering Duration: 4 years Introduction Every life form, beginning from microscopic bacteria to multicellular organisms, can be considered as a…

[academics] — academic_home
  Office of Academics Affairs | IIT Jodhpur ###147852369$$$_RedirectToLoginPage_%%%963258741!!! Welcome to The Office of Academics Academics Facilitates: (a) Admissions Prospectus: preparation & printing; Sale of Application Forms; Issue of Admit Cards for Admission Test; Setting of Test Papers; Condu…

[centre] — CET
  Center for Education Technology (CET) | IIT Jodhpur ###147852369$$$_RedirectToLoginPage_%%%963258741!!! Welcome To The Center For Education Technology (CET) Educational institutions and policymakers around the globe have begun to embrace immersive teaching-learning technologies that will enable most…

[dept

## 11 — How to Load This Corpus in the Word2Vec Notebook

In [15]:
# In your word2vec_iitj.ipynb, replace the raw_docs dict with:

import json
from pathlib import Path

def load_iitj_corpus(json_path='iitj_corpus/all_docs.json') -> dict:
    """
    Load the scraped corpus produced by iitj_scraper.ipynb.
    Returns a dict {label: text} compatible with the Word2Vec notebook.
    """
    with open(json_path, encoding='utf-8') as f:
        docs = json.load(f)
    return {d['label']: d['text'] for d in docs if d.get('text')}

# Usage:
# raw_docs = load_iitj_corpus()
# print(f"Loaded {len(raw_docs)} documents.")

print("Loader function defined. Add this block at the top of word2vec_iitj.ipynb.")

Loader function defined. Add this block at the top of word2vec_iitj.ipynb.
